In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

### **Importing Prerequisites**

In [0]:
%run "/Workspace/Users/0bcr9999@gmail.com/FMGC-DATAENGINEERING-PROJECT/notebooks/setup/3. utilities"

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text("catalog", "fmcg")
dbutils.widgets.text("data_source", "", "Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

print(catalog, data_source)

### **Read Data from Bronze Layer**

In [0]:
df_bronze = spark.sql("SELECT * FROM fmcg.bronze.products")
#df_bronze = spark.read.table("fmcg.bronze.products")

In [0]:
display(df_bronze)

## **Transformations**

**1. Duplicate handling**

In [0]:
df_dup = display(df_bronze.groupBy("product_id","product_name","category").agg(F.count("*")))

In [0]:
print("Count of rows before dropping duplicates: ", df_bronze.count())
df_silver = df_bronze.dropDuplicates()
print("Count of rows after dropping duplicates: ", df_bronze.count())

**2. Title case fix**

In [0]:
df_silver.select("category").distinct().show()

In [0]:
df_silver = df_silver.withColumn(
    "category", 
    F.when(F.col("category").isNull(), None)
    .otherwise(F.initcap("category"))
    )

In [0]:
df_silver.select("category").distinct().show()

**3. Fix the spelling mistake `protien`**

In [0]:
df_silver = (
    df_silver
    .withColumn("product_name", 
            F.regexp_replace(F.col("product_name"), "(?i)protien", "protein")
            )
    .withColumn("category",
                        F.regexp_replace(F.col("category"), "(?i)protien", "protein")
                        )
)


In [0]:
display(df_silver)

**4. Standardizing Customer Attributes to match parent company data model**

In [0]:
df_silver = (
  df_silver
  .withColumn("division",
              F.when(F.col("category") == "Energy Bars", "Nutrition Bars")
               .when(F.col("category") == "Protien Bars", "Nutrition Bars")
               .when(F.col("category") == "Granola & Cereals", "Breakfast Foods")
               .when(F.col("category") == "Recovery Dairy", "Dairy & Recovery")       
               .when(F.col("category") == "Healthy Snacks", "Healthy Snacks")
               .when(F.col("category") == "Electrolyte Mix", "Hydration & Electrolytes")
               .otherwise("Other")       
              )
)

**5. Extracting Variant column from product_name**

In [0]:
df_silver = (
    df_silver.withColumn("variant", F.regexp_extract(F.col("product_name"), r"\((.*?)\)", 1))
)

**6. Clean Product Id - for invalid values keep `99999999`**

In [0]:
df_silver = (
    df_silver
    .withColumn("product_id",
                F.when(F.col("product_id").cast("string").rlike("^[0-9]+$"), 
                       F.col("product_id").cast("string"))
                .otherwise(F.lit(99999999).cast("string"))
                )
)

**7.** **Creating** **product_code** **using hashing function as product id is having invalid values** 

In [0]:
df_silver = (
    df_silver
    .withColumn("product_code", 
                F.sha2(F.col("product_name").cast("string"), 256)
                )
)

**8. Renaming column product_name**

In [0]:
df_silver = df_silver.withColumnRenamed("product_name", "product")

In [0]:
display(df_silver.limit(5))

**9. Re-ordering the columns**

In [0]:
df_silver = df_silver.select("product_code", "division", "category", "product", "variant", "product_id", "dataload_ts", "file_name", "file_size")

In [0]:
display(df_silver.limit(5))

### **writing dataframe to silver table**

In [0]:
df_silver.write \
  .format("delta") \
    .option("delta.enablechangeDataFeed", True) \
      .option("mergeSchema", True) \
        .mode("overwrite") \
          .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")
